# 21 - Pass 10 Q4: does MLM pretraining learn sarcasm or shortcuts? (CPU only, no GPU)

Trains nothing: on Kaggle set **Accelerator: None**. About 10-20 minutes. Run it **after** `results/pass10_stack/`
(notebook 19) and `results/pass10_seeds/` (notebook 20) are committed and pushed; rows whose runs are missing
are skipped.

**Inputs:** only `gainsplit-processed` (for the iSarcasmEval test texts, used by the word-count baseline).
**Internet: on.**

| Part | Question |
|---|---|
| A. Outlet shortcut | Cue-mask gap (random-mask minus cue-mask macro-F1) of the pretrained arms minus that of A1 (A7 for FGM arms). Negative = leans less on outlet cue words. |
| B. Transfer net of length | iSarcasmEval AUC of the pretrained arms minus the AUC of **word count alone** on the same items. |
| C. All Pass 10 verdicts | Q1-Q4 in one list, read from the committed tables of notebooks 19 and 20 and this notebook. |

Contrasts fixed before running in `src/gainsplit/analysis_pass10.py` (vault: Experiment Plan, "Pass 10 plan").

## 1. Get the code and the data

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
import pandas as pd
pd.set_option("display.width", 200)
from gainsplit.data import find_data_dir, verify
from gainsplit import analysis_pass10 as a10
print("code commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
isarc_test = pd.read_csv(Path(DATA_DIR) / "isarcasm_test.csv")
R = Path(REPO_DIR) / "results"
RESULTS_DIRS = [R / p for p in ("pass1", "pass2a", "pass2b", "pass3", "pass4", "pass8_mlm", "pass8_jepa", "pass9",
                                "pass10_stack", "pass10_seeds")]
print("missing (rows that need them are skipped):", [p.name for p in RESULTS_DIRS if not (p / "runs").is_dir()])
RESULTS_DIRS = [p for p in RESULTS_DIRS if (p / "runs").is_dir()]
OUT_DIR = Path("/kaggle/working/analysis_pass10")

## 2. Shortcut gap and transfer net of length (paired bootstrap, 10,000 resamples, Holm within each table)

In [ ]:
tables = a10.run_all(RESULTS_DIRS, "shortcuts", OUT_DIR, isarc_test=isarc_test)
for name in ("summary", "gap_contrasts", "auc_vs_length"):
    if name in tables:
        print(f"\n=== {name}")
        display(tables[name])

## 3. Every Pass 10 verdict in one list
Q1 and Q2/Q3 verdicts come from the tables notebooks 19 and 20 committed; Q4 from this notebook.
Paste this cell's output when you report.

In [ ]:
parts = [pd.read_csv(f) for f in (R / "pass10_stack/analysis/verdicts.csv", R / "pass10_seeds/analysis/verdicts.csv")
         if f.exists()] + [tables["verdicts"]]
v = pd.concat(parts, ignore_index=True)
v.to_csv(OUT_DIR / "all_verdicts.csv", index=False)
for _, r in v.iterrows():
    if pd.isna(r.get("mean_diff")):
        print(f"{r.hypothesis} {r.contrast} at {r.budget}%: {r.verdict}")
        continue
    print(f"{r.hypothesis:6s} {r.contrast:44s} {int(r.budget):3d}%  {r.mean_diff:+.2f} [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  -> {r.verdict}")
print(shutil.make_archive("/kaggle/working/analysis_pass10", "zip", OUT_DIR))